<img src="https://github.com/nicholasmetherall/digital-earth-pacific-macblue-activities/blob/main/attachments/images/DE_Pacific_banner.JPG?raw=true" width="900"/>

Figure 1.1.a. Jupyter environment + Python notebooks

# Digital Earth Pacific Notebook 1 prepare postcard and load data to csv

The objective of this notebook is to prepare a geomad postcard for your AOI (masking, scaling and loading additional band ratios and spectral indices) and sampling all the datasets into a csv based on your training data geodataframe.

## Step 1.1: Configure the environment

In [ ]:
from datetime import datetime
import geopandas as gpd
import pandas as pd
import joblib
import xarray as xr
import numpy as np
from utils import process_year

In [ ]:
# Reload scripts and imports
%load_ext autoreload
%autoreload 2

In [ ]:
# Predefined variable for title and version

# Enter your initials
initials = "agl"

# Enter your site name
site = "tongatapu"

# Date
date = datetime.now()

# Make a clean version string
version = f"{initials}-{site}-{date.strftime('%d%m%Y')}"
print(version)

## Step 1.2: Configure STAC access and search parameters

In [ ]:
# Use training data bounds
training = gpd.read_file("training-data/nm-tongatapu-11122025_postcard_4.geojson")
training = training.to_crs("EPSG:4326")
min_lon, min_lat, max_lon, max_lat = training.total_bounds

bbox = [min_lon, min_lat, max_lon, max_lat]

In [ ]:
# Change this to the name of the class column
training.explore(column="LULC_class")

In [ ]:
# Print values of LULC_code and LULC_class together to see the mapping
training[['LULC_code', 'LULC_class']].drop_duplicates().sort_values(by='LULC_code')

In [ ]:
training.columns.unique()

In [ ]:
model = joblib.load("models/nm-tongatapu-11122025-test.model")

In [ ]:
# If you want to save time later, run this once, then use the cache cell below to save/load

years = range(2017, 2026)
classifications = {}

for year in years:
    predictions, _ = process_year(year, bbox, model, version)
    if predictions is not None:
        classifications[year] = predictions

In [ ]:
# Create a single xarray DataArray to hold all classifications
arrays = []

for year, da in classifications.items():
    da = da.expand_dims(time=[year])
    arrays.append(da)

classifications_ds = xr.combine_by_coords(arrays, combine_attrs="override").to_dataset(name="classification")

In [ ]:
# Use this cell to cache the previous load step.

# Dump to file, to save time later
# classifications_ds.to_zarr(f"classifications_{version}.zarr", mode="w")

# Load from tile, to save time later
# classifications_ds = xr.open_zarr(f"classifications_{version}.zarr")
# classifications_ds

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import colors as mcolors

# Updated classes list to include No Data (Code 0)
lulc_classes = [
    [1, 'Forest_land', '#064a00'],
    [2, 'Grazing_Cropland', '#FFEE8C' ],
    [3, 'Wetland', '#73ffd2'],
    [4, 'Settlements', '#bd0007'],
    [5, 'Bare_land','#919191'],
    [6, 'Water','#71a8ff'],
]

values_list = [c[0] for c in lulc_classes]
color_list = [c[2] for c in lulc_classes]  # Named distinctively

# Use the imported module to build the colormap from your list
cmap = mcolors.ListedColormap(color_list) 

# If you use the bounds/norm later, call them from the module too:
bounds = values_list + [values_list[-1] + 1]
norm = mcolors.BoundaryNorm(bounds, cmap.N)

# cmap = c_map

fig, axes = plt.subplots(3, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, year in enumerate(classifications_ds.time.values):
    ax = axes[idx]
    data = classifications_ds.sel(time=year).squeeze().classification.values
    im = ax.imshow(data, cmap=cmap, vmin=1, vmax=len(lulc_classes))
    ax.set_title(f"LULC {pd.to_datetime(year).year}", fontsize=14, fontweight='bold')
    ax.axis('off')

cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
cbar = fig.colorbar(im, cax=cbar_ax, ticks=range(1, len(lulc_classes)+1))
cbar.ax.set_yticklabels([lulc_classes[i][1] for i in range(len(lulc_classes))])
cbar.set_label('LULC Class', fontsize=12)

plt.suptitle('LULC Classification Time Series - Tongatapu', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Calculate pixel counts for each class per year
change_data = {}

for year in classifications_ds.time.values:
    classification = classifications_ds.sel(time=year).classification
    unique, counts = np.unique(classification.values, return_counts=True)
    change_data[year] = dict(zip(unique, counts))

df_change = pd.DataFrame(change_data).T.fillna(0)

# 1. Create your dictionary mapping
lulc_dict = {c[0]: c[1] for c in lulc_classes}

# 2. Map the columns ONCE (Make sure to remove the duplicate line below it)
df_change.columns = [lulc_dict.get(int(col), f"Class {col}") for col in df_change.columns]

# Convert pixel counts to area (e.g., if 10m resolution, multiply by 100 m²)
pixel_size_m2 = 100  # 10m x 10m pixels
df_area = df_change * pixel_size_m2 / 10000  # Convert to hectares
df_area.index.name = 'Year'

# Plot 1: Stacked area chart
fig, ax = plt.subplots(figsize=(12, 6))
df_area.plot(kind='area', stacked=True, ax=ax, alpha=0.7)
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Area (hectares)', fontsize=12)
ax.set_title('LULC Change Detection - Tongatapu', fontsize=14, fontweight='bold')
ax.legend(title='LULC Class', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

for col in df_area.columns:
    ax.plot(df_area.index, df_area[col], marker='o', linewidth=2.5, markersize=8, label=col)

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Area (hectares)', fontsize=12)
ax.set_title('LULC Class Evolution - Tongatapu', fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='best')
ax.grid(alpha=0.3)
plt.tight_layout()
# plt.savefig('outputs/lulc_trends.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Calculate year-to-year absolute changes (hectares)
df_changes = df_area.diff().fillna(0)

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(df_changes.T, cmap='RdBu_r', aspect='auto', interpolation='nearest')

ax.set_xticks(range(len(df_changes)))
ax.set_xticklabels(df_changes.index)
ax.set_yticks(range(len(df_changes.columns)))
ax.set_yticklabels(df_changes.columns)
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('LULC Class', fontsize=12)
ax.set_title('Year-to-Year LULC Change (hectares)', fontsize=14, fontweight='bold')

cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Change (ha)', fontsize=11)

# Add values in cells
for i in range(len(df_changes)):
    for j in range(len(df_changes.columns)):
        text = ax.text(i, j, f'{df_changes.iloc[i, j]:.0f}',
                      ha="center", va="center", color="black", fontsize=9)

plt.tight_layout()
# plt.savefig('outputs/lulc_change_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
def calculate_transition_matrix(classification_t0, classification_t1, lulc_classes):
    """Calculate transitions between two years"""
    n_classes = len(lulc_classes)
    transition = np.zeros((n_classes, n_classes))
    
    # Flatten arrays
    t0_flat = classification_t0.values.flatten()
    t1_flat = classification_t1.values.flatten()
    
    # Valid pixels (not masked)
    valid = ~(np.isnan(t0_flat) | np.isnan(t1_flat))
    t0_flat = t0_flat[valid].astype(int)
    t1_flat = t1_flat[valid].astype(int)
    
    # Build transition matrix
    for i in range(len(t0_flat)):
        from_class = t0_flat[i] - 1
        to_class = t1_flat[i] - 1
        if 0 <= from_class < n_classes and 0 <= to_class < n_classes:
            transition[from_class, to_class] += 1
    
    return transition

# Calculate transition from 2023 to 2024 (example)
transition_matrix = calculate_transition_matrix(
    classifications_ds.sel(time=2023), 
    classifications_ds.sel(time=2024), 
    lulc_classes
)

# Exclude from/to same class transitions (diagonal) if needed
np.fill_diagonal(transition_matrix, 0)

# Plot transition matrix
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(transition_matrix, cmap='YlOrRd', interpolation='nearest')

class_names = [c[1] for c in lulc_classes]
ax.set_xticks(range(len(class_names)))
ax.set_yticks(range(len(class_names)))
ax.set_xticklabels(class_names, rotation=45, ha='right')
ax.set_yticklabels(class_names)
ax.set_xlabel('To (2024)', fontsize=12, fontweight='bold')
ax.set_ylabel('From (2023)', fontsize=12, fontweight='bold')
ax.set_title('LULC Transition Matrix 2023→2024', fontsize=14, fontweight='bold')

# Add pixel counts
for i in range(len(class_names)):
    for j in range(len(class_names)):
        text = ax.text(j, i, f'{int(transition_matrix[i, j])}',
                      ha="center", va="center", color="black", fontsize=10, fontweight='bold')

cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Pixel Count', fontsize=11)

plt.tight_layout()
# plt.savefig('outputs/transition_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Summary statistics (excluding water)

df_area_no_water = df_area.drop(columns=['Water'], errors='ignore')

summary_stats = pd.DataFrame({
    'Year': classifications_ds.time.values,
    'Total Area (ha)': [df_area_no_water.loc[year].sum() for year in classifications_ds.time.values],
    'Dominant Class': [df_area_no_water.loc[year].idxmax() for year in classifications_ds.time.values],
    'Dominant % of Total': [(df_area_no_water.loc[year].max() / df_area_no_water.loc[year].sum() * 100) for year in classifications_ds.time.values]
})

print(summary_stats.to_string(index=False))